# 12 — Cross-Validation Mastery (Solutions)

## Problem Definition
Estimate private leaderboard performance robustly under finite-sample uncertainty and split constraints.

## Required Prior Knowledge
- Notebooks 10-11 feature-map design and leakage-safe estimation habits.
- Basic estimators and train/validation splitting.

## New Concepts Introduced
- Empirical risk estimator definition.
- Bias and variance of CV estimators.
- KFold/StratifiedKFold/GroupKFold/TimeSeriesSplit.
- Nested CV.
- Out-of-fold prediction protocol.
- Leakage inflation and leaderboard shake simulation.

## Formal Definition
Empirical risk:
$$
\hat{R}(f)=\frac{1}{n}\sum_{i=1}^n L\big(y_i,f(x_i)\big).
$$
K-fold estimator:
$$
\hat{R}_{\mathrm{CV}}(f)=\frac{1}{K}\sum_{k=1}^K
\frac{1}{|V_k|}\sum_{i\in V_k}L\big(y_i,f^{(-k)}(x_i)\big).
$$

Bias-variance decomposition of the estimator:
$$
\mathbb{E}\big[(\hat{R}_{\mathrm{CV}}-R)^2\big]
=\big(\mathbb{E}[\hat{R}_{\mathrm{CV}}]-R\big)^2
+\mathrm{Var}(\hat{R}_{\mathrm{CV}}).
$$

Nested CV objective:
$$
\hat{R}_{\text{nested}}
=\frac{1}{K_{\text{outer}}}\sum_{k=1}^{K_{\text{outer}}}
L\big(y_{V_k}, f_{\lambda_k^*}^{(-k)}(x_{V_k})\big),\quad
\lambda_k^*=\arg\min_{\lambda\in\Lambda}\hat{R}_{\text{inner}}^{(k)}(\lambda).
$$

## Variables and Assumptions
- Scope: Cross-Validation Mastery targets the objective `Estimate private leaderboard performance robustly under finite-sample uncertainty and split constraints.`.
- Data are indexed by $i\in\{1,\dots,n\}$ with features $x_i\in\mathbb{R}^d$ and target $y_i$.
- Model parameters are represented by $\theta$, and predictions are $\hat{y}_i=f_\theta(x_i)$.
- Any preprocessing statistics are fit on training partitions only and reused on validation/test partitions.
- Split protocol (KFold/Group/TimeSeries) must match the data-generating assumptions for IID, grouped, or temporal samples.
- Reported metrics are empirical estimates with finite-sample variance; interpretation must include uncertainty.

## Symbol-by-Symbol Explanation
| Symbol | Meaning |
|---|---|
| $x_i$ | feature vector for sample $i$ |
| $y_i$ | target for sample $i$ |
| $f_\theta$ | model parameterized by $\theta$ |
| $L(\cdot,\cdot)$ | per-sample loss function |
| $n$ | number of samples |
| $d$ | raw feature dimension |
| $p$ | transformed feature dimension / polynomial degree context |
| $K$ | number of folds / partitions |
| $V_k$ | validation index set for fold $k$ |
| $\lambda,\alpha,\tau$ | regularization/smoothing/confidence hyperparameters |

## Zero-Skip Derivation
1. Partition index set $\{1,\dots,n\}$ into disjoint folds $V_1,\dots,V_K$.
2. For each fold $k$, fit model on complement $T_k=\{1,\dots,n\}\setminus V_k$.
3. Compute fold loss average:
   $$
   \hat{R}_k=\frac{1}{|V_k|}\sum_{i\in V_k}L(y_i,f^{(-k)}(x_i)).
   $$
4. Average fold estimates:
   $$
   \hat{R}_{\mathrm{CV}}=\frac{1}{K}\sum_{k=1}^{K}\hat{R}_k.
   $$
5. For leakage inflation, if $\hat{R}_{\text{leaky}}<\hat{R}_{\text{clean}}$ due invalid preprocessing, inflation is
   $$
   \Delta_{\text{leak}}=\hat{R}_{\text{clean}}-\hat{R}_{\text{leaky}}>0.
   $$

## Explicit Logical Transitions
0. Context anchor: this notebook focuses on `Cross-Validation Mastery` and objective `Estimate private leaderboard performance robustly under finite-sample uncertainty and split constraints.`.
1. Start from the formal objective and identify estimators/transformations introduced in this notebook.
2. Map each estimator term to computable quantities under train/validation split constraints.
3. Show why each derivation step is valid (algebraic identity, estimator definition, or probabilistic assumption).
4. Convert the derivation into an implementation protocol with explicit leakage controls.
5. Validate with synthetic and real-data experiments, then interpret failure conditions.

## Intuition
CV is a Monte Carlo-like estimator of deployment risk; splitter choice encodes assumptions about label balance, groups, and temporal ordering.

## Mapping from Math to Implementation
- `make_splitter` selects split strategy.
- `oof_cv_predictions` implements strict out-of-fold protocol.
- `cv_bias_variance_decomposition` summarizes estimator stability.
- `simulate_public_private_variance` approximates leaderboard shake.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

from feature_utils import (
    set_global_seed,
    standardize_train_valid,
    monotone_log1p,
    one_hot_basis,
    polynomial_basis,
    add_interaction_columns,
    target_encode_oof,
    conditional_expectation_feature,
)
from cv_utils import (
    empirical_risk,
    make_splitter,
    oof_cv_predictions,
    cv_bias_variance_decomposition,
    leakage_inflation,
    simulate_public_private_variance,
)
from ensemble_utils import (
    blend_predictions,
    bagging_variance_formula,
    ensemble_variance_from_covariance,
    oof_stacking,
    stacking_predict,
)
from experiment_logger import ExperimentLogger, ExperimentRecord

SEED = 42
set_global_seed(SEED)

MODULE_DIR = Path('.').resolve()

## Synthetic Experiment

In [ ]:
from sklearn.datasets import make_classification

X_syn, y_syn = make_classification(
    n_samples=3000, n_features=20, n_informative=8, n_redundant=4, random_state=SEED, weights=[0.7, 0.3]
)
groups = (np.arange(len(y_syn)) // 30).astype(int)
time_index = np.arange(len(y_syn))

model = LogisticRegression(max_iter=2000)
for split_kind in ["kfold", "stratified", "group", "timeseries"]:
    splitter = make_splitter(split_kind, n_splits=5, seed=SEED)
    if split_kind == "group":
        out = oof_cv_predictions(model, X_syn, y_syn, splitter, task="classification", metric="auc", groups=groups)
    else:
        out = oof_cv_predictions(model, X_syn, y_syn, splitter, task="classification", metric="auc")
    print(split_kind, {"mean_auc": out.mean, "std_auc": out.std})

## Real Dataset Experiment

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data.values
y = (wine.target.values == 0).astype(int)

outer = make_splitter("stratified", n_splits=5, seed=SEED)
outer_scores = []
for tr_idx, va_idx in outer.split(X, y):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Inner model selection (small grid) = nested CV core idea.
    candidates = [0.1, 1.0, 5.0]
    inner = make_splitter("stratified", n_splits=4, seed=SEED)
    best_c, best_inner = None, -1
    for c in candidates:
        clf = LogisticRegression(max_iter=2000, C=c)
        inner_out = oof_cv_predictions(clf, X_tr, y_tr, inner, task="classification", metric="auc")
        if inner_out.mean > best_inner:
            best_inner, best_c = inner_out.mean, c

    final = LogisticRegression(max_iter=2000, C=best_c).fit(X_tr, y_tr)
    auc = roc_auc_score(y_va, final.predict_proba(X_va)[:, 1])
    outer_scores.append(auc)

print({"nested_auc_mean": float(np.mean(outer_scores)), "nested_auc_std": float(np.std(outer_scores))})

## Diagnostic Analysis

In [ ]:
splitter = make_splitter("stratified", n_splits=5, seed=SEED)
out = oof_cv_predictions(LogisticRegression(max_iter=2000), X_syn, y_syn, splitter, task="classification", metric="auc")
diag = cv_bias_variance_decomposition(out.fold_scores)
print(diag)

def auc_metric(y_true, y_pred):
    return roc_auc_score(y_true, y_pred)

shake = simulate_public_private_variance(y_syn, out.oof_pred, auc_metric, public_fraction=0.5, n_trials=200, seed=SEED)
print("public-private shake diagnostics:", shake)

## Failure Analysis

In [ ]:
# Failure case: leakage by fitting scaler once on all data before folds.
scaler = StandardScaler().fit(X_syn)
X_bad = scaler.transform(X_syn)
bad_out = oof_cv_predictions(LogisticRegression(max_iter=2000), X_bad, y_syn, make_splitter("stratified", 5, SEED), task="classification", metric="auc")

# Clean: scaler fit inside fold.
clean_scores = []
splitter = make_splitter("stratified", 5, SEED)
for tr_idx, va_idx in splitter.split(X_syn, y_syn):
    sc = StandardScaler().fit(X_syn[tr_idx])
    Xtr = sc.transform(X_syn[tr_idx]); Xva = sc.transform(X_syn[va_idx])
    clf = LogisticRegression(max_iter=2000).fit(Xtr, y_syn[tr_idx])
    clean_scores.append(roc_auc_score(y_syn[va_idx], clf.predict_proba(Xva)[:, 1]))

print({"leaky_auc": bad_out.mean, "clean_auc": float(np.mean(clean_scores)), "inflation": leakage_inflation(float(np.mean(clean_scores)), bad_out.mean)})

## Exercise Ladder (basic → advanced → research-level)
1. Derive variance reduction trend as K increases under independent fold errors.
2. Show when GroupKFold is strictly necessary.
3. Implement repeated CV and compare estimator variance.
4. Simulate rank instability under tiny public split size.

## Solution Notes
- Verify deterministic behavior by re-running all cells with the same seed and matching key metrics.
- Confirm that no fold-level preprocessing leaks validation targets/features into training statistics.
- Compare synthetic-vs-real conclusions and report where assumptions diverge.

## Summary of Mathematical Insights
Cross-validation is an estimator design problem: split assumptions, nesting, and leakage controls determine whether local gains transfer to private leaderboard reality.